<a href="https://colab.research.google.com/github/PeterJemley/ArtofStatistics/blob/master/modular_analog_clock_sync_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
modular_analog_clock_sync.py

Modular Analog Clock Synchronization Library

A modular, testable analog-clock library providing:
- Immutable Time value object for parsing and arithmetic
- Abstract HandInterface and concrete implementations (MinuteHand, HourHand)
- Dependency injection for clock components
- Pythonic properties and single-responsibility modular design
- Ability to compute minute-hand rotation to synchronize clock times
- Command-line interface (CLI) for direct use as `sync-clocks`

Console script entry point (setup.py / pyproject.toml):
    sync-clocks = modular_analog_clock_sync:main

Author: Peter Jemley, MSHI
"""

from __future__ import annotations
import argparse
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Tuple, Union


# -----------------------------------------------------------------------------
# Value Object: Time
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class Time:
    """
    Immutable value object representing minutes since midnight.

    Attributes:
        total_minutes (int): Total minutes since midnight (0 <= total_minutes < 1440)
    """
    total_minutes: int

    @staticmethod
    def parse(time_str: str) -> Time:
        """
        Parse a time string in the format "[H]H:MM AM|PM" into a Time object.

        Args:
            time_str (str): Time string, e.g. "10:15 AM", "12:00 PM".

        Returns:
            Time: A new Time instance with total_minutes set.

        Raises:
            ValueError: If the format is invalid or falls outside 00:00–23:59 range.

        Examples:
            >>> Time.parse("10:15 AM").total_minutes
            615
            >>> Time.parse("12:00 AM").total_minutes  # Midnight
            0
            >>> Time.parse("12:00 PM").total_minutes  # Noon
            720
        """
        parts = time_str.strip().split()
        if len(parts) != 2:
            raise ValueError(f"Invalid time format: {time_str!r}")
        hhmm, ampm = parts
        hour_str, minute_str = hhmm.split(":")
        hour = int(hour_str)
        minute = int(minute_str)
        ampm = ampm.upper()
        if ampm not in ("AM", "PM"):
            raise ValueError(f"Invalid AM/PM part: {ampm!r}")

        # Normalize hour to 0-23
        if hour == 12:
            hour = 0 if ampm == "AM" else 12
        elif ampm == "PM":
            hour += 12

        total = hour * 60 + minute
        if not 0 <= total < 1440:
            raise ValueError(f"Time out of range: {time_str!r}")
        return Time(total)

    def difference_to(self, other: Time) -> int:
        """
        Compute the forward difference in minutes from this Time to another,
        wrapping around midnight. A zero difference yields 1440 (full day).

        Args:
            other (Time): The target Time.

        Returns:
            int: Minutes to advance clockwise from self to other.

        Examples:
            >>> Time(615).difference_to(Time(765))
            150
            >>> Time(930).difference_to(Time(930))  # Same time
            1440
        """
        raw = (other.total_minutes - self.total_minutes) % 1440
        return raw if raw != 0 else 1440


# -----------------------------------------------------------------------------
# Hand Interfaces & Implementations
# -----------------------------------------------------------------------------

class HandInterface(ABC):
    """
    Abstract base class for clock hands.

    Subclasses must implement 'angle' and 'advance'.
    """

    @property
    @abstractmethod
    def angle(self) -> float:
        """
        Current angle in degrees from 12:00 position (0°..360°).
        """
        ...

    @abstractmethod
    def advance(self, minutes: int) -> None:
        """
        Rotate the hand forward by the specified whole minutes.

        Args:
            minutes (int): Non-negative number of minutes to advance.

        Raises:
            ValueError: If minutes is negative.
        """
        ...


class MinuteHand(HandInterface):
    """
    Minute hand that rests only on perfect minute marks (0..59).
    Snaps any numeric input to the nearest integer minute boundary.
    """

    def __init__(self, minutes: Union[int, float] = 0) -> None:
        """Initialize the MinuteHand, snapping to valid minute."""
        self._minutes = 0
        self.minutes = minutes

    @property
    def minutes(self) -> int:
        """Current minute mark (0..59)."""
        return self._minutes

    @minutes.setter
    def minutes(self, value: Union[int, float]) -> None:
        """
        Snap provided value to nearest integer minute and store mod 60.

        Args:
            value (int|float): Desired minute position (any numeric).
        """
        snapped = int(round(float(value))) % 60
        self._minutes = snapped

    @property
    def angle(self) -> float:
        """Return the minute-hand angle: 6° per minute."""
        return self._minutes * 6.0

    def advance(self, minutes: int) -> None:
        """
        Advance the minute hand clockwise by a whole number of minutes.

        Args:
            minutes (int): Minutes to advance (must be >= 0).

        Raises:
            ValueError: If minutes is negative.
        """
        if minutes < 0:
            raise ValueError("Can only advance forward by non-negative minutes")
        self._minutes = (self._minutes + minutes) % 60


class HourHand(HandInterface):
    """
    Hour hand with continuous movement: 30° per hour plus fraction from minutes.
    """

    def __init__(self, hours: int = 12, minutes: int = 0) -> None:
        """Initialize HourHand at a given 12-hour time position."""
        self._hours = 0
        self._minutes = 0
        self.set_time(hours, minutes)

    def set_time(self, hours: int, minutes: int) -> None:
        """
        Set the hour hand to a given 12-hour time.

        Args:
            hours (int): Hour (1..12).
            minutes (int): Minutes (0..59) to determine fractional angle.

        Raises:
            ValueError: If hours or minutes out of valid range.
        """
        if not 1 <= hours <= 12:
            raise ValueError("Hours must be in 1..12")
        if not 0 <= minutes < 60:
            raise ValueError("Minutes must be in 0..59")
        self._hours = hours % 12
        self._minutes = minutes

    @property
    def angle(self) -> float:
        """
        Return the hour-hand angle: 30° per hour plus 0.5° per minute.
        """
        return (self._hours + self._minutes / 60.0) * 30.0

    def advance(self, minutes: int) -> None:
        """
        Advance the hour hand clockwise by whole minutes of time.

        Args:
            minutes (int): Minutes to advance (must be >= 0).

        Raises:
            ValueError: If minutes is negative.
        """
        if minutes < 0:
            raise ValueError("Can only advance forward by non-negative minutes")
        total = (self._hours * 60 + self._minutes + minutes) % 720
        h, m = divmod(total, 60)
        self._hours = h
        self._minutes = m


# -----------------------------------------------------------------------------
# AnalogClock: Composition with Dependency Injection
# -----------------------------------------------------------------------------

class AnalogClock:
    """
    High-level analog clock composed of minute and hour hands.

    Allows setting time via string and retrieving hand angles.
    """

    def __init__(
        self,
        minute_hand: HandInterface,
        hour_hand: HandInterface
    ) -> None:
        """
        Initialize with injected hand implementations.

        Args:
            minute_hand (HandInterface): Concrete minute-hand instance.
            hour_hand (HandInterface): Concrete hour-hand instance.
        """
        self.minute_hand = minute_hand
        self.hour_hand = hour_hand

    def set_time(self, time_str: str) -> None:
        """
        Parse a time string and position both hands accordingly.

        Args:
            time_str (str): "[H]H:MM AM|PM" format.

        Raises:
            ValueError: If time_str is invalid.
        """
        t = Time.parse(time_str)
        hours_24, minute = divmod(t.total_minutes, 60)
        hour_12 = hours_24 % 12 or 12

        # Position minute hand
        if isinstance(self.minute_hand, MinuteHand):
            self.minute_hand.minutes = minute
        else:
            delta = minute - getattr(self.minute_hand, "minutes", 0)
            self.minute_hand.advance(delta)

        # Position hour hand
        if isinstance(self.hour_hand, HourHand):
            self.hour_hand.set_time(hour_12, minute)
        else:
            total_minutes = (hour_12 % 12) * 60 + minute
            self.hour_hand.advance(total_minutes)

    @property
    def angles(self) -> Tuple[float, float]:
        """
        Get the current angles of hour and minute hands.

        Returns:
            Tuple[float, float]: (hour_hand_angle, minute_hand_angle).
        """
        return self.hour_hand.angle, self.minute_hand.angle


# -----------------------------------------------------------------------------
# ClockSynchronizer: Core Logic
# -----------------------------------------------------------------------------

class ClockSynchronizer:
    """
    Utility class to compute minute-hand rotation needed to synchronize two times.
    """

    @staticmethod
    def degrees_to_travel(time1: str, time2: str) -> float:
        """
        Compute degrees the minute hand must rotate clockwise to move from time1 to time2.

        Args:
            time1 (str): Starting time in "[H]H:MM AM|PM" format.
            time2 (str): Target time in same format.

        Returns:
            float: Degrees to rotate (>= 0).

        Raises:
            ValueError: If input time strings are invalid.

        Examples:
            >>> ClockSynchronizer.degrees_to_travel("10:15 AM", "12:45 PM")
            900.0
        """
        t1 = Time.parse(time1)
        t2 = Time.parse(time2)
        diff_minutes = t1.difference_to(t2)
        return diff_minutes * 6.0  # 6° per minute


# -----------------------------------------------------------------------------
# Command-Line Interface
# -----------------------------------------------------------------------------

def main() -> None:
    """
    CLI entry point for the sync-clocks script.

    Prompts the user if arguments are omitted.

    Usage examples:
        $ sync-clocks "10:15 AM" "12:45 PM"
        900

        $ sync-clocks
        Start time (e.g. 10:15 AM): 10:00 PM
        Target time (e.g. 12:45 PM): 9:00 PM
        8280
    """
    parser = argparse.ArgumentParser(
        description="Compute the minute-hand rotation (degrees) to synchronize two analog clock times."
    )
    parser.add_argument(
        "time1", nargs="?", help="Start time, e.g. '10:15 AM'"
    )
    parser.add_argument(
        "time2", nargs="?", help="Target    time, e.g. '12:45 PM'"
    )
    args = parser.parse_args()

    # Prompt interactively if missing
    if not args.time1:
        args.time1 = input("Start time (e.g. 10:15 AM): ").strip()
    if not args.time2:
        args.time2 = input("Target time (e.g. 12:45 PM): ").strip()

    # Compute and output
    degrees = ClockSynchronizer.degrees_to_travel(args.time1, args.time2)
    print(f"{degrees:.0f}")


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] [time1] [time2]
colab_kernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2